In [28]:
import pandas as pd
import numpy as np

path = "data/ml-100k"

ratings = pd.read_csv(
    f"{path}/u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

genre_cols_raw = [
    "unknown", "Action", "Adventure", "Animation", "Children's",
    "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
    "Film-Noir", "Horror", "Musical", "Mystery", "Romance",
    "Sci-Fi", "Thriller", "War", "Western"
]

item_cols = [
    "item_id", "title", "release_date", "video_release_date", "imdb_url"
] + genre_cols_raw

items = pd.read_csv(
    f"{path}/u.item",
    sep="|",
    names=item_cols,
    encoding="latin-1"
)

# remove unknown
genre_cols = [g for g in genre_cols_raw if g != "unknown"]

genre2idx = {g: k for k, g in enumerate(genre_cols)}
G = len(genre_cols)  # MovieLens 100k: 18

def item_genre_indices(row):
    return [
        genre2idx[g]
        for g in genre_cols
        if row[g] == 1
    ]

items["genre_ids"] = items.apply(item_genre_indices, axis=1)

item_genres = dict(zip(items["item_id"], items["genre_ids"]))

warm start temporal split

In [29]:
def split_by_user_time(df, train_ratio=0.8, val_ratio=0.1, min_n=5):
    train, val, test = [], [], []

    for uid, d in df.sort_values("timestamp").groupby("user_id"):
        d = d.sort_values("timestamp")
        n = len(d)

        if n < min_n:
            train.append(d)
            continue

        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        train.append(d.iloc[:n_train])
        val.append(d.iloc[n_train:n_train+n_val])
        test.append(d.iloc[n_train+n_val:])

    return (
        pd.concat(train).reset_index(drop=True),
        pd.concat(val).reset_index(drop=True),
        pd.concat(test).reset_index(drop=True),
    )

train, val, test = split_by_user_time(ratings)

estimate mu, bi, bu

In [30]:
mu = train["rating"].mean()

user_mean = train.groupby("user_id")["rating"].mean()
item_mean = train.groupby("item_id")["rating"].mean()

b_u = (user_mean - mu).to_dict()
b_i = (item_mean - mu).to_dict()

def add_residual(df):
    df = df.copy()
    df["bu_stat"] = df["user_id"].map(b_u).fillna(0.0)
    df["bi_stat"] = df["item_id"].map(b_i).fillna(0.0)
    df["E"] = df["rating"] - mu - df["bu_stat"] - df["bi_stat"]
    return df

train = add_residual(train)
val = add_residual(val)
test = add_residual(test)
train

,user_id,item_id,rating,timestamp,bu_stat,bi_stat,E
0,1,168,5,874965478,0.013742,0.463228,0.942302
1,1,172,5,874965478,0.013742,0.595029,0.810501
2,1,165,5,874965518,0.013742,0.604457,0.801073
3,1,156,4,874965556,0.013742,0.411459,-0.005929
4,1,166,5,874965677,0.013742,0.536919,0.868611
...,...,...,...,...,...,...,...
79614,943,139,1,888640027,0.016287,-0.685991,-1.911024
79615,943,426,4,888640027,0.016287,-0.455728,0.858713
79616,943,720,1,888640048,0.016287,-0.484574,-2.112441
79617,943,80,2,888640048,0.016287,-0.657651,-0.939364


User taste T_u

In [31]:
from collections import defaultdict
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def build_user_taste_latent_clusters(
    train,
    item_genres,
    G,
    mu,
    K=8,
    beta=10.0,
    min_count=2,
    seed=42
):
    """
    Build latent user type T_u by clustering users' residual preference vectors.

    For each user u, construct a G-dimensional vector:
        x[u, g] = shrinked average residual on genre g

    Then cluster users into K latent types.
    """

    user_sum = defaultdict(lambda: np.zeros(G, dtype=float))
    user_count = defaultdict(lambda: np.zeros(G, dtype=float))

    for row in train.itertuples():
        gs = item_genres.get(row.item_id, [])
        if len(gs) == 0:
            continue

        residual = row.rating - mu - row.bu_stat - row.bi_stat

        w = 1.0 / len(gs)

        for g in gs:
            user_sum[row.user_id][g] += w * residual
            user_count[row.user_id][g] += w

    users = np.array(sorted(train["user_id"].unique()))
    X = np.zeros((len(users), G), dtype=float)

    for idx, uid in enumerate(users):
        sums = user_sum[uid]
        counts = user_count[uid]

        avg_residual = np.divide(
            sums,
            counts,
            out=np.zeros(G, dtype=float),
            where=counts > 0
        )

        shrink = counts / (counts + beta)
        x = shrink * avg_residual

        # genres with too few observations are treated as unknown / neutral
        x[counts < min_count] = 0.0

        X[idx] = x

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    km = KMeans(
        n_clusters=K,
        random_state=seed,
        n_init=20
    )

    labels = km.fit_predict(X_scaled)

    user_T = {
        int(uid): int(label)
        for uid, label in zip(users, labels)
    }

    return user_T, X, labels, km


K = 8

user_T_latent, user_X, user_labels, km = build_user_taste_latent_clusters(
    train=train,
    item_genres=item_genres,
    G=G,
    mu=mu,
    K=K,
    beta=10.0,
    min_count=2,
    seed=42
)

residual bins

In [32]:
B = 8

edges = np.quantile(train["E"], np.linspace(0, 1, B + 1))
edges[0] -= 1e-6
edges[-1] += 1e-6
centers = 0.5 * (edges[:-1] + edges[1:])

def add_E_bin(df):
    df = df.copy()
    df["E_bin"] = np.digitize(df["E"].values, edges[1:-1])
    return df

train = add_E_bin(train)
val = add_E_bin(val)
test = add_E_bin(test)

build p(E|T,g)

In [33]:
def build_prior_E_given_Tg_general(
    train,
    item_genres,
    user_T,
    n_T,
    G,
    B,
    alpha=50.0
):
    """
    Build prior P(E | T, g).

    Shape:
        prior_TgE: (n_T, G, B)

    n_T:
        number of user types.
        For genre-based T, n_T = G.
        For latent clusters, n_T = K.
    """

    counts = np.zeros((n_T, G, B), dtype=float)

    for row in train.itertuples():
        T = user_T.get(row.user_id, 0)
        gs = item_genres.get(row.item_id, [])
        e = int(row.E_bin)

        if len(gs) == 0:
            continue

        w = 1.0 / len(gs)

        for g in gs:
            counts[T, g, e] += w

    global_E = counts.sum(axis=(0, 1))
    global_E = global_E / global_E.sum()

    prior = np.zeros_like(counts)

    for T in range(n_T):
        for g in range(G):
            n = counts[T, g].sum()
            prior[T, g] = (counts[T, g] + alpha * global_E) / (n + alpha)

    return prior

prior_TgE_latent = build_prior_E_given_Tg_general(
    train=train,
    item_genres=item_genres,
    user_T=user_T_latent,
    n_T=K,
    G=G,
    B=B,
    alpha=50.0
)

prior_TgE_latent.shape
# classical conditional probability table / prior table

(8, 18, 8)

## Model

In [34]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class MF(nn.Module):
    def __init__(self, n_users, n_items, k=64, mu=3.5):
        super().__init__()
        self.P = nn.Embedding(n_users, k)
        self.Q = nn.Embedding(n_items, k)
        self.bu = nn.Embedding(n_users, 1)
        self.bi = nn.Embedding(n_items, 1)
        self.mu = nn.Parameter(torch.tensor(float(mu)))

        nn.init.normal_(self.P.weight, std=0.05)
        nn.init.normal_(self.Q.weight, std=0.05)
        nn.init.zeros_(self.bu.weight)
        nn.init.zeros_(self.bi.weight)

    def forward(self, u, i):
        dot = (self.P(u) * self.Q(i)).sum(dim=1)
        return self.mu + self.bu(u).squeeze(1) + self.bi(i).squeeze(1) + dot

In [35]:
class MFDataset(Dataset):
    def __init__(self, df, user2idx, item2idx, user_T, item_genres, G):
        self.u_raw = df["user_id"].values
        self.i_raw = df["item_id"].values

        self.u = df["user_id"].map(user2idx).values.astype("int64")
        self.i = df["item_id"].map(item2idx).values.astype("int64")
        self.r = df["rating"].values.astype("float32")

        self.bu_stat = df["bu_stat"].values.astype("float32")
        self.bi_stat = df["bi_stat"].values.astype("float32")

        self.T = np.array(
            [user_T.get(u, 0) for u in self.u_raw],
            dtype="int64"
        )

        genre_masks = []

        for item_id in self.i_raw:
            mask = np.zeros(G, dtype="float32")
            gs = item_genres.get(item_id, [])

            if len(gs) > 0:
                for g in gs:
                    mask[g] = 1.0 / len(gs)
            else:
                mask[0] = 1.0

            genre_masks.append(mask)

        self.genre_mask = np.stack(genre_masks)

    def __len__(self):
        return len(self.r)

    def __getitem__(self, idx):
        return {
            "u": torch.tensor(self.u[idx]).long(),
            "i": torch.tensor(self.i[idx]).long(),
            "r": torch.tensor(self.r[idx]).float(),
            "bu_stat": torch.tensor(self.bu_stat[idx]).float(),
            "bi_stat": torch.tensor(self.bi_stat[idx]).float(),
            "T": torch.tensor(self.T[idx]).long(),
            "genre_mask": torch.tensor(self.genre_mask[idx]).float(),
        }

In [36]:
def soft_bin(x, centers, tau=0.1):
    # x: [batch]
    # centers: [B]
    dist2 = (x[:, None] - centers[None, :]) ** 2
    return torch.softmax(-dist2 / tau, dim=1)


def prior_loss(pred_rating, batch, mu_fixed, centers_t, prior_TgE_t, tau=0.1, eps=1e-8):
    """
    prior_TgE_t: [T=18, g=18, E=8]
    genre_mask: [batch, 18], each row sums to 1
    """

    pred_E = pred_rating - mu_fixed - batch["bu_stat"] - batch["bi_stat"]
    q_E = soft_bin(pred_E, centers_t, tau=tau)  # [batch, B]

    T = batch["T"]  # [batch]
    prior_for_T = prior_TgE_t[T]  # [batch, G, B]

    # Average genre priors according to movie genre mask
    # p_E[b] = sum_g genre_mask[g] * p(E | T, g)
    p_E = torch.einsum("bg,bge->be", batch["genre_mask"], prior_for_T)

    loss = -(q_E * torch.log(p_E + eps)).sum(dim=1).mean()
    return loss

## Training

In [37]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd

In [38]:
user_ids = sorted(train["user_id"].unique())
item_ids = sorted(train["item_id"].unique())

user2idx = {u: idx for idx, u in enumerate(user_ids)}
item2idx = {i: idx for idx, i in enumerate(item_ids)}

n_users = len(user2idx)
n_items = len(item2idx)

def keep_known_users_items(df, user2idx, item2idx):
    return df[
        df["user_id"].isin(user2idx) &
        df["item_id"].isin(item2idx)
    ].reset_index(drop=True)

train_eval = keep_known_users_items(train, user2idx, item2idx)
val_eval = keep_known_users_items(val, user2idx, item2idx)
test_eval = keep_known_users_items(test, user2idx, item2idx)

print(len(train), len(train_eval))
print(len(val), len(val_eval))
print(len(test), len(test_eval))

79619 79619
9596 9558
10785 10736


In [39]:
# DataLoader

batch_size = 1024

train_ds = MFDataset(train_eval, user2idx, item2idx, user_T_latent, item_genres, G)
val_ds = MFDataset(val_eval, user2idx, item2idx, user_T_latent, item_genres, G)
test_ds = MFDataset(test_eval, user2idx, item2idx, user_T_latent, item_genres, G)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

In [40]:
# Evaluation function：RMSE / MAE

@torch.no_grad()
def evaluate_model(model, loader, clip=True):
    device = next(model.parameters()).device
    model.eval()

    preds = []
    trues = []

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        pred = model(batch["u"], batch["i"])

        if clip:
            pred = torch.clamp(pred, 1.0, 5.0)

        preds.append(pred.cpu().numpy())
        trues.append(batch["r"].cpu().numpy())

    preds = np.concatenate(preds)
    trues = np.concatenate(trues)

    rmse = np.sqrt(np.mean((preds - trues) ** 2))
    mae = np.mean(np.abs(preds - trues))

    return {
        "rmse": rmse,
        "mae": mae,
    }

In [41]:
def train_model(
    model,
    train_loader,
    val_loader=None,
    prior_TgE=None,
    lambda_q=0.0,
    epochs=20,
    lr=1e-3,
    weight_decay=1e-5,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    opt = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    mse_fn = nn.MSELoss()

    centers_t = torch.tensor(centers, dtype=torch.float32, device=device)
    mu_fixed = torch.tensor(float(mu), dtype=torch.float32, device=device)

    if prior_TgE is not None:
        prior_TgE_t = torch.tensor(prior_TgE, dtype=torch.float32, device=device)
    else:
        prior_TgE_t = None

    history = []

    for epoch in range(1, epochs + 1):
        model.train()

        total_mse = 0.0
        total_prior = 0.0
        total_loss = 0.0

        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            pred = model(batch["u"], batch["i"])

            mse = mse_fn(pred, batch["r"])
            loss = mse

            pl = torch.tensor(0.0, device=device)

            if prior_TgE_t is not None and lambda_q > 0:
                pl = prior_loss(
                    pred,
                    batch,
                    mu_fixed=mu_fixed,
                    centers_t=centers_t,
                    prior_TgE_t=prior_TgE_t,
                    tau=0.1,
                )
                loss = mse + lambda_q * pl

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_mse += mse.item()
            total_prior += pl.item()
            total_loss += loss.item()

        avg_mse = total_mse / len(train_loader)
        avg_prior = total_prior / len(train_loader)
        avg_loss = total_loss / len(train_loader)

        row = {
            "epoch": epoch,
            "train_mse": avg_mse,
            "train_rmse": np.sqrt(avg_mse),
            "train_prior": avg_prior,
            "train_loss": avg_loss,
        }

        if val_loader is not None:
            val_metrics = evaluate_model(model, val_loader, clip=True)
            row["val_rmse"] = val_metrics["rmse"]
            row["val_mae"] = val_metrics["mae"]

            print(
                f"epoch {epoch:02d} | "
                f"train_rmse={row['train_rmse']:.4f} | "
                f"prior={avg_prior:.4f} | "
                f"val_rmse={row['val_rmse']:.4f} | "
                f"val_mae={row['val_mae']:.4f}"
            )
        else:
            print(
                f"epoch {epoch:02d} | "
                f"train_rmse={row['train_rmse']:.4f} | "
                f"prior={avg_prior:.4f}"
            )

        history.append(row)

    history = pd.DataFrame(history)

    return model, history

-------------------

In [42]:
def run_one_experiment(
    exp_name,
    user_T_exp,
    lambda_q,
    alpha=50.0,
    epochs=20,
    seed=42
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    prior_TgE_exp = build_prior_E_given_Tg_general(
        train=train,
        movie_genres=item_genres,
        user_T=user_T_exp,
        G=G,
        B=B,
        alpha=alpha
    )

    train_ds = MFDataset(train_eval, user2idx, item2idx, user_T_exp, item_genres, G)
    val_ds = MFDataset(val_eval, user2idx, item2idx, user_T_exp, item_genres, G)
    test_ds = MFDataset(test_eval, user2idx, item2idx, user_T_exp, item_genres, G)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    model = MF(
        n_users=n_users,
        n_items=n_items,
        k=64,
        mu=mu
    )

    use_prior = lambda_q > 0

    model, hist = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        prior_TgE=prior_TgE_exp if use_prior else None,
        lambda_q=lambda_q,
        epochs=epochs,
        lr=1e-3,
        weight_decay=1e-5
    )

    val_metrics = evaluate_model(model, val_loader)
    test_metrics = evaluate_model(model, test_loader)

    result = {
        "experiment": exp_name,
        "lambda_q": lambda_q,
        "alpha": alpha,
        "val_rmse": val_metrics["rmse"],
        "val_mae": val_metrics["mae"],
        "test_rmse": test_metrics["rmse"],
        "test_mae": test_metrics["mae"],
        "best_val_rmse": hist["val_rmse"].min(),
        "best_epoch": int(hist.loc[hist["val_rmse"].idxmin(), "epoch"])
    }

    return model, hist, result, prior_TgE_exp

In [43]:
# baseline and prior grid search

lambda_grid = [0.0, 0.001, 0.003, 0.01, 0.03, 0.1]
K_grid = [4, 8, 12, 16]


latent_results = []
latent_histories = {}
latent_models = {}

for K in [4, 8, 12, 16]:
    print("=" * 70)
    print(f"Building latent user types, K = {K}")
    print("=" * 70)

    user_T_latent, user_X, labels, km = build_user_taste_latent_clusters(
        train=train,
        item_genres=item_genres,
        G=G,
        mu=mu,
        K=K,
        beta=10.0,
        min_count=2,
        seed=42
    )

    prior_TgE_latent = build_prior_E_given_Tg_general(
        train=train,
        item_genres=item_genres,
        user_T=user_T_latent,
        n_T=K,
        G=G,
        B=B,
        alpha=50.0
    )

    train_ds = MFDataset(train_eval, user2idx, item2idx, user_T_latent, item_genres, G)
    val_ds = MFDataset(val_eval, user2idx, item2idx, user_T_latent, item_genres, G)
    test_ds = MFDataset(test_eval, user2idx, item2idx, user_T_latent, item_genres, G)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    for lambda_q in lambda_grid:
        print(f"K={K}, lambda_q={lambda_q}")

        torch.manual_seed(42)
        np.random.seed(42)

        model = MF(
            n_users=n_users,
            n_items=n_items,
            k=64,
            mu=mu
        )

        use_prior = lambda_q > 0

        model, hist = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            prior_TgE=prior_TgE_latent if use_prior else None,
            lambda_q=lambda_q,
            epochs=20,
            lr=1e-3,
            weight_decay=1e-5
        )

        val_metrics = evaluate_model(model, val_loader)
        test_metrics = evaluate_model(model, test_loader)

        latent_results.append({
            "K": K,
            "lambda_q": lambda_q,
            "val_rmse": val_metrics["rmse"],
            "val_mae": val_metrics["mae"],
            "test_rmse": test_metrics["rmse"],
            "test_mae": test_metrics["mae"],
            "best_val_rmse": hist["val_rmse"].min(),
            "best_epoch": int(hist.loc[hist["val_rmse"].idxmin(), "epoch"])
        })

        latent_histories[(K, lambda_q)] = hist
        latent_models[(K, lambda_q)] = model

latent_results_df = pd.DataFrame(latent_results).sort_values("val_rmse")
latent_results_df

Building latent user types, K = 4
K=4, lambda_q=0.0
epoch 01 | train_rmse=1.0947 | prior=0.0000 | val_rmse=1.1639 | val_mae=0.9650
epoch 02 | train_rmse=1.0672 | prior=0.0000 | val_rmse=1.1442 | val_mae=0.9464
epoch 03 | train_rmse=1.0399 | prior=0.0000 | val_rmse=1.1244 | val_mae=0.9263
epoch 04 | train_rmse=1.0085 | prior=0.0000 | val_rmse=1.1000 | val_mae=0.9013
epoch 05 | train_rmse=0.9689 | prior=0.0000 | val_rmse=1.0720 | val_mae=0.8701
epoch 06 | train_rmse=0.9232 | prior=0.0000 | val_rmse=1.0417 | val_mae=0.8377
epoch 07 | train_rmse=0.8771 | prior=0.0000 | val_rmse=1.0177 | val_mae=0.8131
epoch 08 | train_rmse=0.8344 | prior=0.0000 | val_rmse=0.9995 | val_mae=0.7958
epoch 09 | train_rmse=0.7952 | prior=0.0000 | val_rmse=0.9875 | val_mae=0.7844
epoch 10 | train_rmse=0.7593 | prior=0.0000 | val_rmse=0.9800 | val_mae=0.7769
epoch 11 | train_rmse=0.7261 | prior=0.0000 | val_rmse=0.9757 | val_mae=0.7723
epoch 12 | train_rmse=0.6949 | prior=0.0000 | val_rmse=0.9735 | val_mae=0.7697


,K,lambda_q,val_rmse,val_mae,test_rmse,test_mae,best_val_rmse,best_epoch
17,12,0.100,0.991144,0.779865,1.033082,0.819557,0.973185,13
11,8,0.100,0.991157,0.779854,1.033150,0.819603,0.973251,13
23,16,0.100,0.991176,0.779827,1.033097,0.819571,0.973226,13
16,12,0.030,0.991232,0.779988,1.033083,0.819605,0.973222,13
10,8,0.030,0.991236,0.779988,1.033101,0.819616,0.973242,13
22,16,0.030,0.991236,0.779972,1.033091,0.819614,0.973233,13
5,4,0.100,0.991250,0.779945,1.033140,0.819630,0.973323,13
4,4,0.030,0.991257,0.780009,1.033096,0.819624,0.973261,13
15,12,0.010,0.991260,0.780028,1.033080,0.819621,0.973235,13
21,16,0.010,0.991261,0.780021,1.033085,0.819625,0.973238,13


In [44]:
baseline_row = latent_results_df[
    latent_results_df["lambda_q"] == 0.0
].sort_values("val_rmse").iloc[0]

best_latent_row = latent_results_df[
    latent_results_df["lambda_q"] > 0
].sort_values("val_rmse").iloc[0]

comparison_latent = pd.DataFrame([
    {
        "experiment": "baseline",
        "K": baseline_row["K"],
        "lambda_q": 0.0,
        "val_rmse": baseline_row["val_rmse"],
        "test_rmse": baseline_row["test_rmse"],
        "test_mae": baseline_row["test_mae"],
    },
    {
        "experiment": "latent_prior",
        "K": best_latent_row["K"],
        "lambda_q": best_latent_row["lambda_q"],
        "val_rmse": best_latent_row["val_rmse"],
        "test_rmse": best_latent_row["test_rmse"],
        "test_mae": best_latent_row["test_mae"],
    }
])

comparison_latent["test_rmse_improvement"] = (
    comparison_latent.loc[0, "test_rmse"] - comparison_latent["test_rmse"]
)

comparison_latent

,experiment,K,lambda_q,val_rmse,test_rmse,test_mae,test_rmse_improvement
0,baseline,16.0,0.0,0.991270,1.033073,0.819625,0.000000
1,latent_prior,12.0,0.1,0.991144,1.033082,0.819557,-0.000009


In [45]:
# # shuffle latent comparison


# best_K = int(best_latent_row["K"])
# best_lambda = float(best_latent_row["lambda_q"])

# user_T_latent, user_X, labels, km = build_user_taste_latent_clusters(
#     train=train,
#     item_genres=item_genres,
#     G=G,
#     mu=mu,
#     K=best_K,
#     beta=10.0,
#     min_count=2,
#     seed=42
# )


# def shuffle_user_T(user_T, seed=42):
#     rng = np.random.default_rng(seed)

#     users = np.array(list(user_T.keys()))
#     values = np.array(list(user_T.values()))

#     shuffled_values = values.copy()
#     rng.shuffle(shuffled_values)

#     return dict(zip(users, shuffled_values))



# user_T_latent_shuffled = shuffle_user_T(user_T_latent, seed=42)

# prior_TgE_latent_shuffled = build_prior_E_given_Tg_general(
#     train=train,
#     item_genres=item_genres,
#     user_T=user_T_latent_shuffled,
#     n_T=best_K,
#     G=G,
#     B=B,
#     alpha=50.0
# )


# train_ds_shuffled = MFDataset(
#     train_eval,
#     user2idx,
#     item2idx,
#     user_T_latent_shuffled,
#     item_genres,
#     G
# )

# val_ds_shuffled = MFDataset(
#     val_eval,
#     user2idx,
#     item2idx,
#     user_T_latent_shuffled,
#     item_genres,
#     G
# )

# test_ds_shuffled = MFDataset(
#     test_eval,
#     user2idx,
#     item2idx,
#     user_T_latent_shuffled,
#     item_genres,
#     G
# )

# train_loader_shuffled = DataLoader(
#     train_ds_shuffled,
#     batch_size=batch_size,
#     shuffle=True
# )

# val_loader_shuffled = DataLoader(
#     val_ds_shuffled,
#     batch_size=batch_size,
#     shuffle=False
# )

# test_loader_shuffled = DataLoader(
#     test_ds_shuffled,
#     batch_size=batch_size,
#     shuffle=False
# )


# torch.manual_seed(42)
# np.random.seed(42)

# shuffled_latent_model = MF(
#     n_users=n_users,
#     n_items=n_items,
#     k=64,
#     mu=mu
# )

# shuffled_latent_model, shuffled_latent_hist = train_model(
#     model=shuffled_latent_model,
#     train_loader=train_loader_shuffled,
#     val_loader=val_loader_shuffled,
#     prior_TgE=prior_TgE_latent_shuffled,
#     lambda_q=best_lambda,
#     epochs=20,
#     lr=1e-3,
#     weight_decay=1e-5
# )


# shuffled_latent_val = evaluate_model(
#     shuffled_latent_model,
#     val_loader_shuffled
# )

# shuffled_latent_test = evaluate_model(
#     shuffled_latent_model,
#     test_loader_shuffled
# )

# print("Shuffled latent val:", shuffled_latent_val)
# print("Shuffled latent test:", shuffled_latent_test)




In [46]:
# baseline_row = latent_results_df[
#     latent_results_df["lambda_q"] == 0.0
# ].sort_values("val_rmse").iloc[0]

# real_latent_row = best_latent_row

# latent_shuffle_compare = pd.DataFrame([
#     {
#         "experiment": "baseline",
#         "K": int(baseline_row["K"]),
#         "lambda_q": 0.0,
#         "val_rmse": baseline_row["val_rmse"],
#         "val_mae": baseline_row["val_mae"],
#         "test_rmse": baseline_row["test_rmse"],
#         "test_mae": baseline_row["test_mae"],
#     },
#     {
#         "experiment": "real_latent_prior",
#         "K": best_K,
#         "lambda_q": best_lambda,
#         "val_rmse": real_latent_row["val_rmse"],
#         "val_mae": real_latent_row["val_mae"],
#         "test_rmse": real_latent_row["test_rmse"],
#         "test_mae": real_latent_row["test_mae"],
#     },
#     {
#         "experiment": "shuffled_latent_prior",
#         "K": best_K,
#         "lambda_q": best_lambda,
#         "val_rmse": shuffled_latent_val["rmse"],
#         "val_mae": shuffled_latent_val["mae"],
#         "test_rmse": shuffled_latent_test["rmse"],
#         "test_mae": shuffled_latent_test["mae"],
#     }
# ])

# baseline_test_rmse = latent_shuffle_compare.loc[
#     latent_shuffle_compare["experiment"] == "baseline",
#     "test_rmse"
# ].iloc[0]

# latent_shuffle_compare["test_rmse_improvement_vs_baseline"] = (
#     baseline_test_rmse - latent_shuffle_compare["test_rmse"]
# )

# latent_shuffle_compare.round(5)

In [47]:
# Five-seed evaluation at the selected Method 2 hyperparameters
import random
import gc

SEEDS = [0, 1, 2, 3, 4]
BEST_K = 12
BEST_LAMBDA = 0.1
ALPHA = 50.0
EPOCHS = 20
CLUSTER_SEED = 42


def set_all_seeds(seed):
    """Set Python, NumPy, and PyTorch random seeds."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Improve reproducibility when CUDA is used.
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def shuffle_user_types(user_T, seed):
    """Permute cluster labels across users while preserving label counts."""
    rng = np.random.default_rng(seed)
    users = np.array(sorted(user_T.keys()))
    labels = np.array([user_T[u] for u in users])
    shuffled_labels = rng.permutation(labels)
    return {int(u): int(t) for u, t in zip(users, shuffled_labels)}


def make_loaders(user_T_exp, seed):
    """Create reproducible train/validation/test DataLoaders."""
    train_ds = MFDataset(
        train_eval, user2idx, item2idx, user_T_exp, item_genres, G
    )
    val_ds = MFDataset(
        val_eval, user2idx, item2idx, user_T_exp, item_genres, G
    )
    test_ds = MFDataset(
        test_eval, user2idx, item2idx, user_T_exp, item_genres, G
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    return train_loader, val_loader, test_loader


def run_fixed_setting(
    experiment,
    user_T_exp,
    seed,
    lambda_q,
    n_T=BEST_K,
    alpha=ALPHA,
    epochs=EPOCHS,
):
    """Train and evaluate one model for one random seed."""
    set_all_seeds(seed)

    train_loader, val_loader, test_loader = make_loaders(
        user_T_exp=user_T_exp,
        seed=seed,
    )

    prior_exp = None
    if lambda_q > 0:
        prior_exp = build_prior_E_given_Tg_general(
            train=train,
            item_genres=item_genres,
            user_T=user_T_exp,
            n_T=n_T,
            G=G,
            B=B,
            alpha=alpha,
        )

    model = MF(
        n_users=n_users,
        n_items=n_items,
        k=64,
        mu=mu,
    )

    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        prior_TgE=prior_exp,
        lambda_q=lambda_q,
        epochs=epochs,
        lr=1e-3,
        weight_decay=1e-5,
    )

    val_metrics = evaluate_model(model, val_loader)
    test_metrics = evaluate_model(model, test_loader)

    result = {
        "experiment": experiment,
        "seed": seed,
        "K": n_T,
        "lambda_q": lambda_q,
        "val_rmse": float(val_metrics["rmse"]),
        "val_mae": float(val_metrics["mae"]),
        "test_rmse": float(test_metrics["rmse"]),
        "test_mae": float(test_metrics["mae"]),
        "best_val_rmse": float(history["val_rmse"].min()),
        "best_epoch": int(history.loc[history["val_rmse"].idxmin(), "epoch"]),
    }

    # Release memory before the next run.
    del model, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# Fit the real K=12 clustering once and keep it fixed across training seeds.
user_T_latent_5seed, user_X_5seed, labels_5seed, km_5seed = (
    build_user_taste_latent_clusters(
        train=train,
        item_genres=item_genres,
        G=G,
        mu=mu,
        K=BEST_K,
        beta=10.0,
        min_count=2,
        seed=CLUSTER_SEED,
    )
)

print("Selected setting:")
print({
    "K": BEST_K,
    "lambda_q": BEST_LAMBDA,
    "alpha": ALPHA,
    "epochs": EPOCHS,
    "seeds": SEEDS,
})

Selected setting:
{'K': 12, 'lambda_q': 0.1, 'alpha': 50.0, 'epochs': 20, 'seeds': [0, 1, 2, 3, 4]}


In [48]:
# Run baseline, real cluster prior, and shuffled cluster prior for five seeds
five_seed_results = []

for seed in SEEDS:
    print("\n" + "=" * 80)
    print(f"Training seed {seed}")
    print("=" * 80)

    # 1. Baseline MF: same initialization and batch-order seed as the two prior models.
    baseline_result = run_fixed_setting(
        experiment="Baseline MF",
        user_T_exp=user_T_latent_5seed,
        seed=seed,
        lambda_q=0.0,
    )
    five_seed_results.append(baseline_result)

    # 2. Real KMeans cluster prior.
    real_result = run_fixed_setting(
        experiment="Real cluster prior",
        user_T_exp=user_T_latent_5seed,
        seed=seed,
        lambda_q=BEST_LAMBDA,
    )
    five_seed_results.append(real_result)

    # 3. Shuffled control: preserve cluster counts, but permute labels across users.
    user_T_shuffled_seed = shuffle_user_types(
        user_T_latent_5seed,
        seed=seed,
    )
    shuffled_result = run_fixed_setting(
        experiment="Shuffled cluster prior",
        user_T_exp=user_T_shuffled_seed,
        seed=seed,
        lambda_q=BEST_LAMBDA,
    )
    five_seed_results.append(shuffled_result)

five_seed_df = pd.DataFrame(five_seed_results)
five_seed_df


Training seed 0
epoch 01 | train_rmse=1.0946 | prior=0.0000 | val_rmse=1.1644 | val_mae=0.9653
epoch 02 | train_rmse=1.0672 | prior=0.0000 | val_rmse=1.1438 | val_mae=0.9466
epoch 03 | train_rmse=1.0403 | prior=0.0000 | val_rmse=1.1243 | val_mae=0.9271
epoch 04 | train_rmse=1.0095 | prior=0.0000 | val_rmse=1.1017 | val_mae=0.9034
epoch 05 | train_rmse=0.9709 | prior=0.0000 | val_rmse=1.0746 | val_mae=0.8734
epoch 06 | train_rmse=0.9251 | prior=0.0000 | val_rmse=1.0454 | val_mae=0.8415
epoch 07 | train_rmse=0.8777 | prior=0.0000 | val_rmse=1.0200 | val_mae=0.8153
epoch 08 | train_rmse=0.8337 | prior=0.0000 | val_rmse=1.0012 | val_mae=0.7970
epoch 09 | train_rmse=0.7940 | prior=0.0000 | val_rmse=0.9893 | val_mae=0.7852
epoch 10 | train_rmse=0.7582 | prior=0.0000 | val_rmse=0.9812 | val_mae=0.7775
epoch 11 | train_rmse=0.7249 | prior=0.0000 | val_rmse=0.9777 | val_mae=0.7732
epoch 12 | train_rmse=0.6939 | prior=0.0000 | val_rmse=0.9759 | val_mae=0.7709
epoch 13 | train_rmse=0.6649 | prio

,experiment,seed,K,lambda_q,val_rmse,val_mae,test_rmse,test_mae,best_val_rmse,best_epoch
0,Baseline MF,0,12,0.0,0.995645,0.783889,1.040003,0.824638,0.975867,13
1,Real cluster prior,0,12,0.1,0.995587,0.783764,1.040013,0.824616,0.975884,13
2,Shuffled cluster prior,0,12,0.1,0.995652,0.783881,1.039949,0.824596,0.975878,12
3,Baseline MF,1,12,0.0,0.996005,0.782872,1.035077,0.822096,0.976277,13
4,Real cluster prior,1,12,0.1,0.995778,0.782677,1.034999,0.822065,0.976198,13
5,Shuffled cluster prior,1,12,0.1,0.995977,0.782844,1.034981,0.821999,0.976302,13
6,Baseline MF,2,12,0.0,0.988643,0.779068,1.025806,0.816263,0.968975,13
7,Real cluster prior,2,12,0.1,0.988571,0.778989,1.025804,0.816254,0.968945,13
8,Shuffled cluster prior,2,12,0.1,0.988658,0.779110,1.025747,0.816187,0.969000,13
9,Baseline MF,3,12,0.0,0.993041,0.780975,1.031422,0.818419,0.974081,13


In [49]:
# Aggregate the five-seed results
experiment_order = [
    "Baseline MF",
    "Real cluster prior",
    "Shuffled cluster prior",
]

five_seed_summary = (
    five_seed_df
    .groupby("experiment", sort=False)
    .agg(
        mean_val_rmse=("val_rmse", "mean"),
        std_val_rmse=("val_rmse", "std"),
        mean_test_rmse=("test_rmse", "mean"),
        std_test_rmse=("test_rmse", "std"),
        mean_test_mae=("test_mae", "mean"),
        std_test_mae=("test_mae", "std"),
        mean_best_epoch=("best_epoch", "mean"),
    )
    .reindex(experiment_order)
    .reset_index()
)

baseline_mean_rmse = five_seed_summary.loc[
    five_seed_summary["experiment"] == "Baseline MF",
    "mean_test_rmse",
].iloc[0]

five_seed_summary["rmse_improvement_vs_baseline"] = (
    baseline_mean_rmse - five_seed_summary["mean_test_rmse"]
)

# Numeric table for subsequent analysis.
five_seed_summary.round(6)

,experiment,mean_val_rmse,std_val_rmse,mean_test_rmse,std_test_rmse,mean_test_mae,std_test_mae,mean_best_epoch,rmse_improvement_vs_baseline
0,Baseline MF,0.991947,0.004273,1.031698,0.006033,0.819582,0.003667,13.0,0.000000
1,Real cluster prior,0.991833,0.004245,1.031713,0.006013,0.819572,0.003664,13.0,-0.000015
2,Shuffled cluster prior,0.991912,0.004312,1.031633,0.006025,0.819527,0.003658,12.8,0.000065


In [50]:
# Paired seed-level comparison
method2_seed_pivot = five_seed_df.pivot(
    index="seed",
    columns="experiment",
    values=["test_rmse", "test_mae"],
)

paired_rmse = five_seed_df.pivot(
    index="seed",
    columns="experiment",
    values="test_rmse",
).reset_index()

paired_rmse["real_gain_vs_baseline"] = (
    paired_rmse["Baseline MF"] - paired_rmse["Real cluster prior"]
)
paired_rmse["shuffled_gain_vs_baseline"] = (
    paired_rmse["Baseline MF"] - paired_rmse["Shuffled cluster prior"]
)
paired_rmse["real_gain_over_shuffled"] = (
    paired_rmse["Shuffled cluster prior"] - paired_rmse["Real cluster prior"]
)

paired_rmse.round(6)

experiment,seed,Baseline MF,Real cluster prior,Shuffled cluster prior,real_gain_vs_baseline,shuffled_gain_vs_baseline,real_gain_over_shuffled
0,0,1.040003,1.040013,1.039949,-0.000011,0.000054,-0.000064
1,1,1.035077,1.034999,1.034981,0.000079,0.000096,-0.000017
2,2,1.025806,1.025804,1.025747,0.000002,0.000059,-0.000057
3,3,1.031422,1.031512,1.031345,-0.000090,0.000077,-0.000168
4,4,1.026183,1.026240,1.026144,-0.000056,0.000039,-0.000095
